In [ ]:
!pip install -U transformers
!pip install -U accelerate
!pip install -U datasets
!pip install -U bertviz
!pip install -U umap-learn
!pip install seaborn --upgrade

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/twitter_multi_class_sentiment.csv")
df.head()

In [ ]:
df['label'].value_counts()

## Data Analysis

In [ ]:
import matplotlib.pyplot as plt

label_counts = df['label_name'].value_counts(ascending=True)
label_counts.plot.barh()
plt.title("Frequency of Classes")
plt.show()

In [ ]:
df['Words per Tweet'] = df['text'].str.split().apply(len)
df.boxplot('Words per Tweet', by='label_name', grid=False, figsize=(10, 6))
df

## Tokenization :
Note : Transformers receives tokenized words post cleaning of text and then further embedding is passed to the transformers model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_ckpt = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
text = "I love fine-tuning of BERT transformer model."

encoded_text = tokenizer(text)
print(encoded_text)


In [ ]:
len(tokenizer.vocab), tokenizer.vocab_size, tokenizer.model_max_length

## Data Loader and Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify = df['label_name'])
test_df, validation = train_test_split(test_df, test_size=1/3, random_state=42, stratify = test_df['label_name'])

train_df.shape, test_df.shape, validation.shape

In [ ]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False),
    'validation': Dataset.from_pandas(validation, preserve_index=False)
})

dataset

## Tokenization of the Emotion/Sentiment Data

In [ ]:
dataset['train'][0]

In [ ]:
def tokenize(batch):
  return tokenizer(batch['text'], padding=True, truncation=True)

print(tokenize(dataset['train'][0]))

In [ ]:
emotion_encoded = dataset.map(tokenize, batched=True, batch_size=None)
emotion_encoded

In [ ]:
label2id = {x['label_name']:x['label'] for x in dataset['train']}
id2label = {v:k for k,v in label2id.items()}

label2id, id2label

## Model Building

In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(model_ckpt)
model

In [ ]:
model.config.architectures

In [ ]:
model.config.id2label

## Loading Model for Classification with Classifier


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoConfig

num_labels = len(label2id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = AutoConfig.from_pretrained(model_ckpt, num_labels=num_labels, id2label=id2label, label2id=label2id)

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, id2label=id2label, label2id=label2id).to(device)
model

In [ ]:
device

In [ ]:
model.config

## Building Training Arguments

In [ ]:
from transformers import TrainingArguments

batch_size = 64
training_dir = "bert_base_train_dir"

training_args = TrainingArguments(
    output_dir=training_dir,
    num_train_epochs=2,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    weight_decay=0.01,
    disable_tqdm=False,
    report_to = []
)

## Building Compute Metrics

In [ ]:
!pip install evaluate

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics_evaluate(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  acc = accuracy_score(labels, preds)
  f1 = f1_score(labels, preds, average='weighted')
  return {'accuracy': acc, 'f1': f1}

## Building Trainer

In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=emotion_encoded['train'],
    eval_dataset=emotion_encoded['validation'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

In [ ]:
trainer.train()

## Model Evaluation

In [ ]:
preds_output = trainer.predict(
    emotion_encoded['test']
)
preds_output.metrics

In [ ]:
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = emotion_encoded['test']['label']

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred))

## Plot Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
cm

In [ ]:
label2id

In [ ]:
sns.heatmap(cm, annot=True, xticklabels=label2id.keys(), yticklabels=label2id.keys(),fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

## Build Prediction Model and Serialize model

In [ ]:
text = "Finally !! I reached out to a girl and asked her out for a date."

In [ ]:
def get_predictions(text):
  inputs = tokenizer(text, return_tensors='pt').to(device)
  with torch.no_grad():
    outputs = model(**inputs)

  logits = outputs.logits
  pred = torch.argmax(logits, dim=1).item()
  return id2label[pred]

In [ ]:
get_predictions(text)

In [ ]:
trainer.save_model("bert-base-uncased-sentiment-defined-model")

## Use Pipeline for Predictions

In [ ]:
from transformers import pipeline
classifer = pipeline("text-classification", model="bert-base-uncased-sentiment-defined-model")
classifer(text)

In [ ]:
text = "Under Operation Sindoor, India killed Islamic terrorists attacking deep down inside of Pakistan territory"
classifer(text)